In [ ]:
import pandas as pd
import os
import sys
import numpy as np
import shutil
from IPython.display import Audio

#speaker diarization
import torch
import torchaudio
from scripts import mycut
from pyannote.audio import Pipeline
from nemo.collections.asr.models import SortformerEncLabelModel

#extracting of features
from scripts import myvoice

In [ ]:
import importlib

homepath=os.getcwd()
scripts_path=os.path.join(homepath, 'scripts')
sys.path.append(scripts_path)

import mycut
import myml
importlib.reload(mycut)
importlib.reload(myml)


In [ ]:
homepath=os.getcwd()
scripts_path=os.path.join(homepath, 'scripts')
sys.path.append(scripts_path)

import myml
import mystress


# Check version of Python==64
!python -c "import sys; print(sys.maxsize > 2**32)"

# random seed for reproducability
np.random.seed(42)

# show all columns of dataframes
pd.set_option('display.max_columns', None)

# set current dir to highest hierachy to add data path
os.chdir('/')
data_folder='/data/'

sys.path.append(data_folder)
os.chdir(homepath)

In [ ]:
main_dir=data_folder + 'raw/recordings'
output_filename='data_new.csv'

In [ ]:
def remove(to_delete=['features.csv']):
    for files in to_delete:
        try:
            os.remove(os.path.join(files))
        except OSError:
            pass

In [ ]:
def get_audio_path(main_dir, output_filename):
    
    main_dir=os.path.join(main_dir)
    df=pd.DataFrame() 
    print(os.listdir(main_dir))
        
    for subdir, dirs, files in os.walk(main_dir):
        for file in files:
            #print os.path.join(subdir, file)
            path = subdir + os.sep + file
            print (path)
            #silence_based_conversion(path)
            break
    return path

In [ ]:
path=get_audio_path(main_dir, output_filename)
#silence_based_conversion(path)

# Get the original (uncut) recordings from the data folder

Here we are seleceting all recordings from the male and female participant folders

In [ ]:
# Root directory that contains the original dataset structure (e.g., data/raw/males, data/raw/females).
BASE_DIR = "data/raw/uncut_data"

# Output directory where selected audio recordings will be copied to, grouped by gender.
RAW_AUDIO_DIR = "data/raw_audios"

# Subdirectories expected under BASE_DIR and created under RAW_AUDIO_DIR.
GROUPS = ["males", "females"]

for group in GROUPS:
    # Path to the current source directory (e.g., data/raw/males).
    group_path = os.path.join(BASE_DIR, group)

    # Create the corresponding target directory (e.g., data/raw_recordings/males).
    target_group_dir = os.path.join(RAW_AUDIO_DIR, group)
    os.makedirs(target_group_dir, exist_ok=True)

    for vp_folder in sorted(os.listdir(group_path)):
        vp_path = os.path.join(group_path, vp_folder)

        if not os.path.isdir(vp_path):
            continue

        for file_name in sorted(os.listdir(vp_path)):

            # Only consider files that match the desired recording pattern.
            # Here: recordings ending with "-1-recording.wav".
            if not file_name.endswith("-1-recording.wav"):
                continue

            # Full path to the source audio file.
            audio_path = os.path.join(vp_path, file_name)

            # Destination path in the group target directory.
            target_path = os.path.join(target_group_dir, file_name)

            try:
                shutil.copy(audio_path, target_path)
                print("Saved:", target_path)

            except Exception as e:
                print(f"Error copying {audio_path}: {e}")

# Speaker diarization with pyannote + Cutting participant speaking parts together

## For one participant

Here we are first performing speaker diarizaion with pyannote and then cutting only the speaker parts from the participant together. We do this for one partcicipant to test it only.

### Perform speaker diarization
Model: https://huggingface.co/pyannote/speaker-diarization-community-1

In [ ]:
audio_path = "path/to/participant_audio.wav"

waveform, sample_rate = torchaudio.load(audio_path)
waveform = waveform.mean(dim=0, keepdim=True) 

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1"
)
pipeline.to(torch.device("cuda"))

output = pipeline({"waveform": waveform, "sample_rate": sample_rate})

for turn, speaker in output.speaker_diarization:
    print(f"{speaker} speaks between t={turn.start:.2f}s and t={turn.end:.2f}s")

### Identify the study participant

In [ ]:
speaker_durations = {}
for turn, speaker in output.speaker_diarization:
    duration = turn.end - turn.start
    if speaker in speaker_durations:
        speaker_durations[speaker] += duration
    else:
        speaker_durations[speaker] = duration

vp_speaker = max(speaker_durations, key=speaker_durations.get)
print(f"The study participant (vp_speaker) is: {vp_speaker}")

### Extracting only the participant speech

In [ ]:
audio_path = "path/to/participant_audio.wav"

out_path, keep_intervals = mycut.cut_only_vp_parts_and_remove_overlaps(
    audio_path=audio_path,
    diarization_output=output,
    vp_speaker=vp_speaker,
    out_path="participant_cleaned.wav",
    eps_ms=50
)

print("Saved:", out_path)

### Listen to cleaned participant recording

In [ ]:
Audio("participant_cleaned.wav")

# Speaker diarization using Sortformer (from NVDIA Neo speached AI) + Cutting participant speach segments together

## For one particpant

Here we are first performing speaker diarizaion with pyannote and then cutting only the speaker parts from the participant together. We do this for one partcicipant to test it only.

In [ ]:
Audio("path/to/participant_audio.wav")

### Perform speaker diarization
Model: https://huggingface.co/nvidia/diar_streaming_sortformer_4spk-v2

In [ ]:
audio_file = "path/to/participant_audio.wav"

diar_model = SortformerEncLabelModel.from_pretrained("nvidia/diar_streaming_sortformer_4spk-v2")
diar_model.eval()

diar_model.sortformer_modules.chunk_len = 340
diar_model.sortformer_modules.chunk_right_context = 40
diar_model.sortformer_modules.fifo_len = 40
diar_model.sortformer_modules.spkcache_update_period = 300

predicted_segments = diar_model.diarize(audio=[audio_file], batch_size=1)

for segment in predicted_segments[0]:
    print(segment)


### Identify the study participant

In [ ]:
speaker_durations = {}
segments = predicted_segments[0]

for line in segments:
    start, end, speaker = line.split()

    duration = float(end) - float(start)

    if speaker in speaker_durations:
        speaker_durations[speaker] += duration
    else:
        speaker_durations[speaker] = duration

vp_speaker = max(speaker_durations, key=speaker_durations.get)

print(f"The study participant (vp_speaker) is: {vp_speaker}")

### Extracting only the participant speech

In [ ]:
audio_path = "path/to/participant_audio.wav"

out_path, clean_vp = mycut.cut_only_vp_parts_and_remove_overlaps_nemo(
    audio_path=audio_path,
    segments=predicted_segments[0],
    vp_speaker=vp_speaker,
    out_path="participant_cleaned.wav",
    eps_ms=50
)

### Listen to cleaned participant recording

In [ ]:
Audio("participant_cleaned.wav")

# Cut all audio files 

Here we are processing the male and female audio recordings with a speaker diarization pipeline. Each recording is first trimmed to a nine-minute segment starting at minute seven. Pyannote identifies the different speakers, we then determine the VP speaker, remove overlapping speech, and save a new WAV file containing only the VP’s speech segments.

In [ ]:
# Base directory containing the raw (uncut) audio files (in subfolders:"males" and "females").
BASE_DIR = "./raw_audios"

# Target base directory where the processed output WAV files will be written.
PROCESSED_BASE = "./processed_wav_pyannote"

# Subfolder names expected inside BASE_DIR.
GROUPS = ["males", "females"]

# Temporary directory used to store trimmed WAV files before diarization and final cutting.
TEMP_DIR = "/tmp_trimmed_wav_pyannote"
os.makedirs(TEMP_DIR, exist_ok=True)  

# Initialize the speaker diarization pipeline.
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
)
pipeline.to(torch.device("cuda"))

for group in GROUPS:
    group_path = os.path.join(BASE_DIR, group)

    for file_name in sorted(os.listdir(group_path)):
        if not file_name.endswith("recording.wav"):
            continue

        audio_path = os.path.join(group_path, file_name)

        # Create the output directory for this group.
        processed_group_dir = os.path.join(PROCESSED_BASE, group)
        os.makedirs(processed_group_dir, exist_ok=True)

        # Build the output filename for the final "VP-only" audio result.
        out_name = file_name.replace("-1-recording.wav", "_Fin_vp_only_pyannote.wav")
        out_path = os.path.join(processed_group_dir, out_name)

        # Build the temp filename for the trimmed segment.
        trimmed_path = os.path.join(
            TEMP_DIR,
            file_name.replace(".wav", "_trimmed_ab7_9.wav")
        )

        # Trim the audio to the relevant segment: starting at minute 7 for a duration of 9 minutes.
        mycut.cut_segment(audio_path, trimmed_path)
        print(f"Trimmed {audio_path} to {trimmed_path}")

        # Run the remaining pipeline on the trimmed audio segment.
        audio_path = trimmed_path

        try:
            waveform, sample_rate = torchaudio.load(audio_path)

            waveform = waveform.mean(dim=0, keepdim=True)

            diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate})

            vp_speaker = mycut.get_vp_speaker(diarization)

            print(f"Processing {audio_path}, vp_speaker: {vp_speaker}")

            # Cut the audio so that only the VP speaker segments remain.
            # and remove overlaps.
            out_path, keep_intervals = mycut.cut_only_vp_parts_and_remove_overlaps(
                audio_path=audio_path,
                diarization_output=diarization,
                vp_speaker=vp_speaker,
                out_path=out_path,
                eps_ms=50
            )

            print("Saved:", out_path)

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")

# Cut all audio files using Sortformer (from NVDIA Neo speached AI)

Here we are processing the male and female audio recordings with a speaker diarization pipeline. Each recording is first trimmed to a nine-minute segment starting at minute seven. NVDIA Softformer identifies the different speakers, we then determine the VP speaker, remove overlapping speech, and save a new WAV file containing only the VP’s speech segments.

In [ ]:
# Base directory containing the raw audio files (in subfolders:"males" and "females").
BASE_DIR = "./raw_audios"

# Target base directory where the processed output WAV files will be written.
PROCESSED_BASE = "./processed_wav_nemo"

# Subfolder names expected inside BASE_DIR.
GROUPS = ["males", "females"]

# Temporary directory used to store trimmed WAV files before diarization and final cutting.
TEMP_DIR = "./tmp_trimmed_wav_nemo"
os.makedirs(TEMP_DIR, exist_ok=True)

# Initialize the speaker diarization model.
diar_model = SortformerEncLabelModel.from_pretrained("nvidia/diar_streaming_sortformer_4spk-v2")
diar_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diar_model.to(device)

# standart parameters (showed on website).
diar_model.sortformer_modules.chunk_len = 340
diar_model.sortformer_modules.chunk_right_context = 40
diar_model.sortformer_modules.fifo_len = 40
diar_model.sortformer_modules.spkcache_update_period = 300

for group in GROUPS:
    group_path = os.path.join(BASE_DIR, group)

    for file_name in sorted(os.listdir(group_path)):
        if not file_name.endswith("recording.wav"):
            continue

        audio_path = os.path.join(group_path, file_name)

        # Create the output directory for this group.
        processed_group_dir = os.path.join(PROCESSED_BASE, group)
        os.makedirs(processed_group_dir, exist_ok=True)

        # Build the output filename for the final "VP-only" audio result.
        out_name = file_name.replace("-1-recording.wav", "_Fin_vp_only_nemo.wav")
        out_path = os.path.join(processed_group_dir, out_name)

        # Build the temp filename for the trimmed segment.
        trimmed_path = os.path.join(
            TEMP_DIR,
            file_name.replace(".wav", "_trimmed_ab7_9.wav")
        )

        # Trim the audio to the relevant segment: starting at minute 7 for a duration of 9 minutes.
        mycut.cut_segment(audio_path, trimmed_path)
        print(f"Trimmed {audio_path} to {trimmed_path}")
        audio_path = trimmed_path

        try:
            predicted_segments = diar_model.diarize(audio=[audio_path], batch_size=1)

            vp_speaker = mycut.get_vp_speaker_nemo(predicted_segments)

            print(f"Processing {audio_path}, vp_speaker: {vp_speaker}")

            # Cut the audio so that only the VP speaker segments remain.
            # and remove overlaps.
            out_path, keep_intervals = mycut.cut_only_vp_parts_and_remove_overlaps_nemo(
                audio_path=audio_path,
                segments=predicted_segments[0],
                vp_speaker=vp_speaker,
                out_path=out_path,
                eps_ms=50
            )

            print("Saved:", out_path)

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")


# Extraction and Merging of Features (one audio per Participant)

Here we are extracting acoustic voice features from the processed recordings using Librosa and Praat. Male and female voices are analyzed separately with gender-specific pitch ranges. Finally, the resulting voice features are combined with the corresponding behavioral data and saved as one merged dataset for further analysis. In this the audios processed from the pipline before (with NVDIA Softformer) are used

In [ ]:
from scripts import myvoice
import pandas as pd
import os
import librosa

main_dir="./processed_wav_nemo"

#LIBROSA
filename_librosa='features.csv'
myvoice.get_librosa_features(main_dir, filename_librosa)


#PRAAT
filename_praat='new_praat_results'

#male Voices
input_path="./processed_wav_nemo/males/*.wav"
output_path=filename_praat+'_male.csv'
f0min=60 #50 #60
f0max=200 #300
myvoice.praat_analysis(input_path, output_path, f0min, f0max)

#female voices
input_path="./processed_wav_nemo/females/*.wav"
output_path=filename_praat+'_female.csv'
f0min=100 #75
f0max=400 #500 
myvoice.praat_analysis(input_path, output_path, f0min, f0max)


#Merge all together
filename_behavior=data_folder + '/TSST_behavior.csv'
filename_output='df_vpn'

myvoice.merge_all_df_together(filename_praat, filename_librosa, filename_behavior, filename_output)

### get opensmile features

Here we are extracting the opensmile features

In [ ]:
input_path="./processed_wav_nemo"
myvoice.get_opensmile_features_regex(input_path, "opensmile_features.csv")